# Stage 1: Generative Pseudo-Thermal Synthesis via Pix2Pix cGAN (Optimized for NVIDIA T4)
### Component C2: Software Emulated Linear Transposed Cross Attention Framework for Noise Resilient Pavement Segmentation

**Project Title:** AI Framework for Sri Lankan Road Management (RoadLens)  
**Research Group:** J26-DS-309  
**Researcher:** Methma C A N (IT23245860) – B.Sc. (Hons) in Information Technology Specializing in Data Science, SLIIT  
**Supervisors:** Ms. Thamali Dassanayake & Ms. Hansi De Silva  

---
### Performance & Optimization Profile on NVIDIA T4 GPU
- **Hardware Acceleration:** Automatic Mixed Precision (PyTorch AMP FP16) utilizing Turing Tensor Cores.
- **I/O Optimization:** Pre-stages images onto Colab local high-speed NVMe SSD (`/content/local_dataset`) to bypass Google Drive FUSE latency.
- **CuDNN Auto-tuner:** Enabled via `torch.backends.cudnn.benchmark = True`.
- **Batch Size:** Scaled to 32 to maximize T4 GPU compute utilization.
- **Throughput Target:** Reduction from ~4.68s/it down to **~0.15s – 0.25s/it (20x-30x speedup)**.

## 1. Hardware & Compute Environment Verification
Verifies active NVIDIA T4 GPU accelerator, enables CuDNN benchmark mode, and checks VRAM.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    # Enable cuDNN benchmark for optimized convolution kernels on fixed input dimensions (256x256)
    torch.backends.cudnn.benchmark = True
    print("cuDNN Benchmark enabled.")
else:
    print("WARNING: CUDA is not available. Please enable GPU via Runtime > Change runtime type > T4 GPU.")

## 2. Dependencies Installation
Installs computer vision and evaluation packages (`scikit-image`, `albumentations`, `opencv-python`, `matplotlib`, `tqdm`).

In [ ]:
!pip install -q scikit-image albumentations opencv-python matplotlib tqdm

## 3. Persistent Storage Mount & High-Speed Local SSD Staging
### **Crucial T4 Speed Optimization:**
Reading thousands of images individually through `/content/drive/MyDrive/...` via Google Drive FUSE causes massive network latency (each file read takes 100-200ms, causing 4.5+ seconds per batch).

We copy/rsync the dataset to the **Colab local NVMe SSD** at `/content/local_dataset` once. Checkpoints and exported models are still saved directly back to **Google Drive** for persistence.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive_mount_point = "/content/drive"
    if not os.path.exists(drive_mount_point):
        drive.mount(drive_mount_point)
        print("Google Drive mounted successfully.")
    else:
        print(f"Google Drive already mounted at {drive_mount_point}.")
    DRIVE_ROOT = Path("/content/drive/MyDrive/C2_Pavement_Research")
except ImportError:
    print("Running locally.")
    DRIVE_ROOT = Path("./C2_Pavement_Research")

# Source dataset in Google Drive
DRIVE_DATASET = DRIVE_ROOT / "datasets" / "IRT_Pavement"
DRIVE_RGB = DRIVE_DATASET / "RGB" if (DRIVE_DATASET / "RGB").exists() else DRIVE_DATASET / "01-Visible images"
DRIVE_THM = DRIVE_DATASET / "thermal" if (DRIVE_DATASET / "thermal").exists() else DRIVE_DATASET / "02-Infrared images"

# Target fast local NVMe SSD directory in Colab VM
LOCAL_DATASET = Path("/content/local_dataset")
LOCAL_RGB = LOCAL_DATASET / "RGB"
LOCAL_THM = LOCAL_DATASET / "thermal"

EXP_DIR = DRIVE_ROOT / "experiments" / "cgan_stage1"
WEIGHTS_DIR = DRIVE_ROOT / "weights"
SAMPLES_DIR = EXP_DIR / "visual_samples"

for d in [EXP_DIR, WEIGHTS_DIR, SAMPLES_DIR, LOCAL_RGB, LOCAL_THM]:
    d.mkdir(parents=True, exist_ok=True)

# Fast stage copy to local SSD (only if not already copied)
num_local_rgb = len(list(LOCAL_RGB.glob("*")))
if num_local_rgb == 0 and DRIVE_RGB.exists():
    print("Copying dataset from Google Drive to local high-speed SSD...")
    # Use fast system rsync / cp
    !cp -r "{DRIVE_RGB}"/* "{LOCAL_RGB}"/
    !cp -r "{DRIVE_THM}"/* "{LOCAL_THM}"/
    print("Data staging complete!")
else:
    print(f"Dataset already staged locally on SSD ({num_local_rgb} images).")

# Set active training directories to the fast local SSD
RGB_DIR = LOCAL_RGB if len(list(LOCAL_RGB.glob("*"))) > 0 else DRIVE_RGB
THERMAL_DIR = LOCAL_THM if len(list(LOCAL_THM.glob("*"))) > 0 else DRIVE_THM

print(f"Active RGB Source:     {RGB_DIR}")
print(f"Active Thermal Source: {THERMAL_DIR}")
print(f"Persistent Weights:    {WEIGHTS_DIR}")

## 4. Dataset Discovery & Route-Based Geographical Partitioning (Proposal Sec 3.6)
Partitions image pairs strictly by collection route/session dates (`YYYYMMDD` prefix) into:
- **~80% Training Set**
- **~20% Validation Set**
This guarantees zero temporal/frame leakage across consecutive video frames.

In [ ]:
import glob
import re
import random
from typing import List, Tuple, Dict

def discover_paired_dataset(rgb_dir: Path, thermal_dir: Path) -> List[Tuple[Path, Path]]:
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    rgb_files = {p.stem: p for p in rgb_dir.iterdir() if p.suffix.lower() in valid_exts}
    thermal_files = {p.stem: p for p in thermal_dir.iterdir() if p.suffix.lower() in valid_exts}
    
    common_stems = sorted(list(set(rgb_files.keys()).intersection(set(thermal_files.keys()))))
    if len(common_stems) == 0:
        norm_rgb = {re.sub(r'^(rgb|thermal|t|vis)[_-]?', '', k, flags=re.IGNORECASE): v for k, v in rgb_files.items()}
        norm_thm = {re.sub(r'^(rgb|thermal|t|vis)[_-]?', '', k, flags=re.IGNORECASE): v for k, v in thermal_files.items()}
        common_norm = sorted(list(set(norm_rgb.keys()).intersection(set(norm_thm.keys()))))
        pairs = [(norm_rgb[k], norm_thm[k]) for k in common_norm]
    else:
        pairs = [(rgb_files[s], thermal_files[s]) for s in common_stems]
        
    print(f"Discovered {len(rgb_files)} RGB files and {len(thermal_files)} Thermal files.")
    print(f"Successfully matched {len(pairs)} co-registered RGB-Thermal pairs.")
    return pairs

pairs = discover_paired_dataset(RGB_DIR, THERMAL_DIR)
assert len(pairs) > 0, f"No pairs found! Verify that {RGB_DIR} and {THERMAL_DIR} contain images."

def split_by_route_session(pairs: List[Tuple[Path, Path]], val_ratio: float = 0.20, seed: int = 42):
    sessions: Dict[str, List[Tuple[Path, Path]]] = {}
    for rgb_path, thm_path in pairs:
        prefix = rgb_path.stem[:8] if len(rgb_path.stem) >= 8 and rgb_path.stem[:8].isdigit() else "default"
        sessions.setdefault(prefix, []).append((rgb_path, thm_path))

    if len(sessions) <= 1:
        rng = random.Random(seed)
        shuffled = pairs.copy()
        rng.shuffle(shuffled)
        split = int((1.0 - val_ratio) * len(shuffled))
        return shuffled[:split], shuffled[split:], {}

    sorted_sessions = sorted(sessions.keys())
    rng = random.Random(seed)
    rng.shuffle(sorted_sessions)

    total_pairs = len(pairs)
    target_val = int(val_ratio * total_pairs)

    val_pairs, train_pairs = [], []
    val_count = 0
    val_sessions = set()
    for s in sorted_sessions:
        s_cnt = len(sessions[s])
        if val_count + s_cnt <= target_val * 1.15 or val_count == 0:
            val_sessions.add(s)
            val_count += s_cnt
        else:
            break

    for s, s_pairs in sessions.items():
        if s in val_sessions:
            val_pairs.extend(s_pairs)
        else:
            train_pairs.extend(s_pairs)

    return train_pairs, val_pairs, sessions

train_pairs, val_pairs, session_breakdown = split_by_route_session(pairs, val_ratio=0.20)

print(f"\nRoute Session Partitioning:")
for s, s_list in sorted(session_breakdown.items()):
    print(f"  - Session {s}: {len(s_list)} image pairs")

print(f"\nTraining set:   {len(train_pairs)} pairs ({len(train_pairs)/len(pairs)*100:.1f}%)")
print(f"Validation set: {len(val_pairs)} pairs ({len(val_pairs)/len(pairs)*100:.1f}%)")

## 5. Optimized Dataset Loader & Fast Optical Noise Augmentations
- Synchronous geometric transforms (flips, rotations) on both RGB and thermal.
- Asymmetric synthetic shadow and glare injected onto the RGB input.
- Scaled to **Batch Size 32** with `num_workers=4` for parallel data fetching.

In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class AsymmetricOpticalNoise:
    """Fast vectorized simulation of tree canopy shadows and solar glare on RGB."""
    
    @staticmethod
    def apply_synthetic_shadow(image: np.ndarray) -> np.ndarray:
        h, w = image.shape[:2]
        num_shadows = np.random.randint(1, 3)
        shadow_mask = np.ones((h, w), dtype=np.float32)
        
        for _ in range(num_shadows):
            num_points = np.random.randint(3, 6)
            points = np.column_stack([
                np.random.randint(0, w, size=num_points),
                np.random.randint(0, h, size=num_points)
            ])
            cv2.fillPoly(shadow_mask, [points], np.random.uniform(0.35, 0.65))
            
        shadow_mask = cv2.GaussianBlur(shadow_mask, (15, 15), 0)
        return (image.astype(np.float32) * shadow_mask[:, :, None]).clip(0, 255).astype(np.uint8)

    @staticmethod
    def apply_solar_glare(image: np.ndarray) -> np.ndarray:
        h, w = image.shape[:2]
        cx, cy = np.random.randint(0, w), np.random.randint(0, h)
        radius = np.random.randint(min(h, w) // 5, min(h, w) // 2)
        
        y, x = np.ogrid[:h, :w]
        dist_sq = (x - cx) ** 2 + (y - cy) ** 2
        glare = np.exp(-dist_sq / (2.0 * (radius / 2.0) ** 2)) * np.random.uniform(120, 190)
        return np.clip(image.astype(np.float32) + glare[:, :, None], 0, 255).astype(np.uint8)


class IRTPavementDataset(Dataset):
    def __init__(self, pairs: List[Tuple[Path, Path]], img_size: int = 256, is_train: bool = True):
        self.pairs = pairs
        self.img_size = img_size
        self.is_train = is_train
        
    def __len__(self) -> int:
        return len(self.pairs)
        
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        rgb_path, thm_path = self.pairs[idx]
        
        rgb_img = cv2.imread(str(rgb_path))
        if rgb_img is None:
            raise ValueError(f"Failed to load image: {rgb_path}")
        rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB)
        
        thm_img = cv2.imread(str(thm_path), cv2.IMREAD_GRAYSCALE)
        if thm_img is None:
            thm_img = cv2.imread(str(thm_path))
            thm_img = cv2.cvtColor(thm_img, cv2.COLOR_BGR2GRAY)
            
        rgb_img = cv2.resize(rgb_img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        thm_img = cv2.resize(thm_img, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        
        if self.is_train:
            if random.random() > 0.5:
                rgb_img = np.fliplr(rgb_img)
                thm_img = np.fliplr(thm_img)
            if random.random() > 0.5:
                rgb_img = np.flipud(rgb_img)
                thm_img = np.flipud(thm_img)
            if random.random() > 0.5:
                k = random.choice([1, 2, 3])
                rgb_img = np.rot90(rgb_img, k)
                thm_img = np.rot90(thm_img, k)
                
            if random.random() > 0.5:
                rgb_img = AsymmetricOpticalNoise.apply_synthetic_shadow(rgb_img)
            if random.random() > 0.6:
                rgb_img = AsymmetricOpticalNoise.apply_solar_glare(rgb_img)
                
        # Fast normalization to [-1, 1]
        rgb_tensor = torch.from_numpy(rgb_img.copy()).permute(2, 0, 1).float().div_(127.5).sub_(1.0)
        thm_tensor = torch.from_numpy(thm_img.copy()).unsqueeze(0).float().div_(127.5).sub_(1.0)
        
        return rgb_tensor, thm_tensor

train_dataset = IRTPavementDataset(train_pairs, img_size=256, is_train=True)
val_dataset = IRTPavementDataset(val_pairs, img_size=256, is_train=False)

# Batch size 32 maximizes T4 GPU compute throughput
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Optimized DataLoaders ready: {len(train_loader)} train batches ({BATCH_SIZE}/batch), {len(val_loader)} validation batches.")

## 6. Inspecting Training Samples & Noise Simulation
Visualizes sample pairs to confirm that the RGB input, synthetic shadow/glare noise, and target thermal maps are properly formatted.

In [ ]:
import matplotlib.pyplot as plt

def denorm(tensor: torch.Tensor) -> np.ndarray:
    img = (tensor.detach().cpu().numpy() + 1.0) / 2.0
    return np.clip(img, 0.0, 1.0)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i in range(2):
    rgb_t, thm_t = train_dataset[i]
    
    rgb_disp = np.transpose(denorm(rgb_t), (1, 2, 0))
    axes[i, 0].imshow(rgb_disp)
    axes[i, 0].set_title(f"Sample {i+1}: RGB (w/ Aug)")
    axes[i, 0].axis("off")
    
    thm_disp = denorm(thm_t).squeeze()
    axes[i, 1].imshow(thm_disp, cmap="inferno")
    axes[i, 1].set_title("Target Thermal (Inferno)")
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(thm_disp, cmap="bone")
    axes[i, 2].set_title("Radiometric Voids (Bone)")
    axes[i, 2].axis("off")
    
    axes[i, 3].hist(thm_disp.flatten(), bins=30, color="teal", alpha=0.7)
    axes[i, 3].set_title("Thermal Intensity Dist.")
    axes[i, 3].set_xlim([0, 1])

plt.tight_layout()
plt.show()

## 7. Pix2Pix cGAN Architecture: UNet-256 Generator & 70x70 PatchGAN Discriminator
UNet Generator with skip connections & 70x70 PatchGAN Discriminator.

In [ ]:
import torch.nn as nn

class UNetDownBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, normalize: bool = True):
        super().__init__()
        layers = [nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=not normalize)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_channels))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UNetUpBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True)
        ]
        if dropout > 0.0:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
        
    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.block(x)
        return torch.cat([x, skip], dim=1)


class UNetGenerator256(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        self.down1 = UNetDownBlock(in_channels, 64, normalize=False)  # 128x128
        self.down2 = UNetDownBlock(64, 128)                           # 64x64
        self.down3 = UNetDownBlock(128, 256)                          # 32x32
        self.down4 = UNetDownBlock(256, 512)                          # 16x16
        self.down5 = UNetDownBlock(512, 512)                          # 8x8
        self.down6 = UNetDownBlock(512, 512)                          # 4x4
        self.down7 = UNetDownBlock(512, 512)                          # 2x2
        self.down8 = UNetDownBlock(512, 512, normalize=False)         # 1x1
        
        self.up1 = UNetUpBlock(512, 512, dropout=0.5)                 # 2x2 (512+512)
        self.up2 = UNetUpBlock(1024, 512, dropout=0.5)                # 4x4 (512+512)
        self.up3 = UNetUpBlock(1024, 512, dropout=0.5)                # 8x8 (512+512)
        self.up4 = UNetUpBlock(1024, 512)                             # 16x16 (512+512)
        self.up5 = UNetUpBlock(1024, 256)                             # 32x32 (256+256)
        self.up6 = UNetUpBlock(512, 128)                              # 64x64 (128+128)
        self.up7 = UNetUpBlock(256, 64)                               # 128x128 (64+64)
        
        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        d6 = self.down6(d5)
        d7 = self.down7(d6)
        d8 = self.down8(d7)
        
        u1 = self.up1(d8, d7)
        u2 = self.up2(u1, d6)
        u3 = self.up3(u2, d5)
        u4 = self.up4(u3, d4)
        u5 = self.up5(u4, d3)
        u6 = self.up6(u5, d2)
        u7 = self.up7(u6, d1)
        return self.final(u7)


class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels: int = 4):
        super().__init__()
        def discriminator_block(in_f: int, out_f: int, normalize: bool = True):
            layers = [nn.Conv2d(in_f, out_f, kernel_size=4, stride=2, padding=1)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_f))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
            
        self.model = nn.Sequential(
            *discriminator_block(in_channels, 64, normalize=False), # 128x128
            *discriminator_block(64, 128),                          # 64x64
            *discriminator_block(128, 256),                         # 32x32
            nn.Conv2d(256, 512, kernel_size=4, stride=1, padding=1),# 31x31
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1)   # 30x30
        )
        
    def forward(self, rgb: torch.Tensor, thermal: torch.Tensor) -> torch.Tensor:
        x = torch.cat([rgb, thermal], dim=1)
        return self.model(x)


def weights_init_normal(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if hasattr(m, "bias") and m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif classname.find("BatchNorm2d") != -1 or classname.find("InstanceNorm2d") != -1:
        if hasattr(m, "weight") and m.weight is not None:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
        if hasattr(m, "bias") and m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = UNetGenerator256(in_channels=3, out_channels=1).to(device)
discriminator = PatchGANDiscriminator(in_channels=4).to(device)

generator.apply(weights_init_normal)
discriminator.apply(weights_init_normal)
print(f"Models loaded onto: {device}")

## 8. Loss Objectives, Optimization & Automatic Mixed Precision (AMP)
Configures `torch.cuda.amp.GradScaler()` to leverage NVIDIA T4 Tensor Cores for 2x faster math with 50% lower memory.

In [ ]:
import math
from skimage.metrics import structural_similarity as compute_ssim

criterion_GAN = nn.MSELoss()
criterion_L1 = nn.L1Loss()
LAMBDA_L1 = 100.0

LR = 0.0002
BETA1 = 0.5
BETA2 = 0.999
optimizer_G = torch.optim.Adam(generator.parameters(), lr=LR, betas=(BETA1, BETA2))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=LR, betas=(BETA1, BETA2))

# Automatic Mixed Precision (AMP) GradScalers for T4 Tensor Cores
scaler_G = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
scaler_D = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# For 2,600+ pairs, 100 epochs is optimal
NUM_EPOCHS = 100
DECAY_EPOCH = 50

def lr_lambda(epoch: int) -> float:
    return 1.0 - max(0, epoch - DECAY_EPOCH) / float(NUM_EPOCHS - DECAY_EPOCH + 1)

scheduler_G = torch.optim.lr_scheduler.LambdaLR(optimizer_G, lr_lambda=lr_lambda)
scheduler_D = torch.optim.lr_scheduler.LambdaLR(optimizer_D, lr_lambda=lr_lambda)

def calculate_psnr(mse_val: float) -> float:
    if mse_val == 0:
        return float("inf")
    return 20.0 * math.log10(2.0 / math.sqrt(mse_val))


def evaluate_validation_set(gen_model, val_loader, device):
    gen_model.eval()
    total_l1 = 0.0
    total_mse = 0.0
    total_ssim = 0.0
    num_samples = 0
    
    with torch.no_grad():
        for rgb, thm in val_loader:
            rgb, thm = rgb.to(device, non_blocking=True), thm.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                fake_thm = gen_model(rgb)
                l1_val = criterion_L1(fake_thm, thm).item()
                mse_val = torch.mean((fake_thm - thm) ** 2).item()
            
            b = rgb.size(0)
            total_l1 += l1_val * b
            total_mse += mse_val * b
            
            real_np = ((thm.cpu().numpy() + 1.0) / 2.0).squeeze(1)
            fake_np = ((fake_thm.cpu().numpy() + 1.0) / 2.0).squeeze(1)
            
            for i in range(b):
                total_ssim += compute_ssim(real_np[i], fake_np[i], data_range=1.0)
            num_samples += b
            
    return total_l1 / num_samples, calculate_psnr(total_mse / num_samples), total_ssim / num_samples

print(f"Optimizers, schedulers, and PyTorch AMP Scaler initialized. Total epochs set to {NUM_EPOCHS}.")

## 9. Chronological Training Loop with AMP Acceleration
Runs with `autocast()` and `GradScaler()` for ultra-fast iterations on T4.

In [ ]:
import time
from tqdm.notebook import tqdm

history = {
    "loss_G": [],
    "loss_D": [],
    "loss_L1": [],
    "val_mae": [],
    "val_psnr": [],
    "val_ssim": []
}

best_val_ssim = -1.0
best_val_mae = float("inf")

print(f"Starting Fast Stage 1 cGAN Training for {NUM_EPOCHS} epochs on {device}...")
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    generator.train()
    discriminator.train()
    
    epoch_loss_G = 0.0
    epoch_loss_D = 0.0
    epoch_loss_L1 = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch [{epoch}/{NUM_EPOCHS}]", leave=False)
    for rgb, thm in pbar:
        rgb = rgb.to(device, non_blocking=True)
        thm = thm.to(device, non_blocking=True)
        batch_size = rgb.size(0)
        
        patch_shape = (batch_size, 1, 30, 30)
        real_label = torch.ones(patch_shape, device=device)
        fake_label = torch.zeros(patch_shape, device=device)
        
        # -----------------------------------
        # 1. Train Discriminator (with AMP)
        # -----------------------------------
        optimizer_D.zero_grad(set_to_none=True)
        
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            pred_real = discriminator(rgb, thm)
            loss_D_real = criterion_GAN(pred_real, real_label)
            
            fake_thm = generator(rgb)
            pred_fake = discriminator(rgb, fake_thm.detach())
            loss_D_fake = criterion_GAN(pred_fake, fake_label)
            
            loss_D = 0.5 * (loss_D_real + loss_D_fake)
            
        scaler_D.scale(loss_D).backward()
        scaler_D.step(optimizer_D)
        scaler_D.update()
        
        # -----------------------------------
        # 2. Train Generator (with AMP)
        # -----------------------------------
        optimizer_G.zero_grad(set_to_none=True)
        
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            pred_fake_for_G = discriminator(rgb, fake_thm)
            loss_G_GAN = criterion_GAN(pred_fake_for_G, real_label)
            loss_G_L1 = criterion_L1(fake_thm, thm)
            loss_G = loss_G_GAN + (LAMBDA_L1 * loss_G_L1)
            
        scaler_G.scale(loss_G).backward()
        scaler_G.step(optimizer_G)
        scaler_G.update()
        
        epoch_loss_G += loss_G.item()
        epoch_loss_D += loss_D.item()
        epoch_loss_L1 += loss_G_L1.item()
        
        pbar.set_postfix({
            "L_D": f"{loss_D.item():.4f}",
            "L_G": f"{loss_G.item():.2f}",
            "L1": f"{loss_G_L1.item():.4f}"
        })
        
    scheduler_G.step()
    scheduler_D.step()
    
    avg_G = epoch_loss_G / len(train_loader)
    avg_D = epoch_loss_D / len(train_loader)
    avg_L1 = epoch_loss_L1 / len(train_loader)
    
    history["loss_G"].append(avg_G)
    history["loss_D"].append(avg_D)
    history["loss_L1"].append(avg_L1)
    
    # Validation & Periodic Visual Logs every 5 epochs
    if epoch % 5 == 0 or epoch == NUM_EPOCHS:
        val_mae, val_psnr, val_ssim = evaluate_validation_set(generator, val_loader, device)
        history["val_mae"].append(val_mae)
        history["val_psnr"].append(val_psnr)
        history["val_ssim"].append(val_ssim)
        
        print(f"--> [Epoch {epoch:03d}/{NUM_EPOCHS}] Loss_G: {avg_G:.2f} | Loss_D: {avg_D:.4f} | "
              f"Val MAE: {val_mae:.4f} | Val PSNR: {val_psnr:.2f} dB | Val SSIM: {val_ssim:.4f}")
        
        generator.eval()
        with torch.no_grad():
            sample_rgb, sample_thm = next(iter(val_loader))
            sample_rgb = sample_rgb.to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                sample_fake = generator(sample_rgb)
                
            fig, ax = plt.subplots(4, 3, figsize=(12, 14))
            for k in range(min(4, sample_rgb.size(0))):
                ax[k, 0].imshow(np.transpose(denorm(sample_rgb[k]), (1, 2, 0)))
                ax[k, 0].set_title("Input RGB")
                ax[k, 0].axis("off")
                
                ax[k, 1].imshow(denorm(sample_thm[k]).squeeze(), cmap="inferno")
                ax[k, 1].set_title("Real Thermal")
                ax[k, 1].axis("off")
                
                ax[k, 2].imshow(denorm(sample_fake[k]).squeeze(), cmap="inferno")
                ax[k, 2].set_title("Synthesized Pseudo-Thermal")
                ax[k, 2].axis("off")
                
            plt.suptitle(f"Epoch {epoch} Visual Evaluation - Pseudo-Thermal Synthesis", fontsize=14)
            plt.tight_layout()
            plt.savefig(SAMPLES_DIR / f"val_epoch_{epoch:03d}.png", dpi=150)
            plt.close()
            
        if val_ssim > best_val_ssim:
            best_val_ssim = val_ssim
            best_val_mae = val_mae
            torch.save({
                "epoch": epoch,
                "generator_state_dict": generator.state_dict(),
                "discriminator_state_dict": discriminator.state_dict(),
                "val_ssim": val_ssim,
                "val_psnr": val_psnr,
                "val_mae": val_mae
            }, WEIGHTS_DIR / "cgan_generator_best.pth")
            print(f"    * Best model checkpoint saved to Google Drive! (SSIM: {val_ssim:.4f}, MAE: {val_mae:.4f})")

total_mins = (time.time() - start_time) / 60.0
print(f"Training complete in {total_mins:.1f} mins. Peak Val SSIM: {best_val_ssim:.4f}")

## 10. Training Convergence & Metric Plots
Plots loss curves and validation metrics (SSIM & PSNR) for use in your thesis dissertation and viva slides.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history["loss_G"], label="L_Generator", color="darkblue")
axes[0].plot(history["loss_L1"], label="L1 Reconstruction", color="orange")
axes[0].plot(history["loss_D"], label="L_Discriminator", color="green")
axes[0].set_title("Pix2Pix Loss Convergence")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

eval_epochs = list(range(5, NUM_EPOCHS + 1, 5))
axes[1].plot(eval_epochs, history["val_ssim"], marker="o", color="crimson", linewidth=2)
axes[1].set_title("Validation Structural Similarity (SSIM)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("SSIM Score")
axes[1].grid(True, alpha=0.3)

axes[2].plot(eval_epochs, history["val_psnr"], marker="s", color="purple", linewidth=2)
axes[2].set_title("Validation PSNR (dB)")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Peak SNR (dB)")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(EXP_DIR / "training_curves.png", dpi=200)
plt.show()
print(f"Plots saved to: {EXP_DIR / 'training_curves.png'}")

## 11. Freeze & Export "Software Sensor" Generator for Stage 2
Saves `cgan_frozen_sensor.pth` and `cgan_sensor_metadata.json` for the Stage 2 MobileNetV3 + LTCA segmentation network.

In [ ]:
import json

best_checkpoint = torch.load(WEIGHTS_DIR / "cgan_generator_best.pth", map_location="cpu")
frozen_generator = UNetGenerator256(in_channels=3, out_channels=1)
frozen_generator.load_state_dict(best_checkpoint["generator_state_dict"])
frozen_generator.eval()

for param in frozen_generator.parameters():
    param.requires_grad = False

torch.save(frozen_generator.state_dict(), WEIGHTS_DIR / "cgan_frozen_sensor.pth")

metadata = {
    "component": "C2_Stage1_PseudoThermal_cGAN",
    "model_architecture": "UNetGenerator256",
    "input_channels": 3,
    "input_resolution": [256, 256],
    "output_channels": 1,
    "output_resolution": [256, 256],
    "normalization": "RGB in [-1, 1] -> Pseudo-Thermal in [-1, 1]",
    "trained_on": "IRT_Pavement (co-registered pairs)",
    "best_epoch": best_checkpoint["epoch"],
    "val_ssim": best_checkpoint["val_ssim"],
    "val_psnr": best_checkpoint["val_psnr"],
    "val_mae": best_checkpoint["val_mae"],
    "status": "FROZEN_SOFTWARE_SENSOR"
}

with open(WEIGHTS_DIR / "cgan_sensor_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Stage 1 Frozen Software Sensor exported successfully!")
print(f"Weights:  {WEIGHTS_DIR / 'cgan_frozen_sensor.pth'}")
print(f"Metadata: {WEIGHTS_DIR / 'cgan_sensor_metadata.json'}")
print("\nReady for Stage 2 core segmentation training!")